# Résoudre Lunar Lander avec PPO

Ce notebook présente la construction, l’entraînement et l’évaluation d’un agent d’apprentissage par renforcement capable de piloter un atterrisseur dans `LunarLander-v3`.

L’objectif est double : obtenir une politique qui atterrit correctement et comprendre les mécanismes qui rendent **PPO (Proximal Policy Optimization)** robuste dans un environnement physique continu dans ses états et discret dans ses actions.

Nous allons suivre une progression pédagogique :

1. découvrir l’environnement et ses contraintes physiques ;
2. représenter la politique avec un **Actor** et estimer la valeur des états avec un **Critic** ;
3. collecter des trajectoires avec un **rollout buffer** ;
4. calculer les avantages avec **GAE** ;
5. mettre à jour la politique avec l’objectif PPO et son clipping ;
6. entraîner, sauvegarder et évaluer l’agent ;
7. enregistrer une exécution sous forme de vidéo.

> **Question directrice :** comment modifier une politique progressivement, sans détruire les comportements déjà appris ?

## 1. Préparer l’environnement de travail

Les imports regroupent les outils nécessaires à l’expérience :

- **Gymnasium** fournit `LunarLander-v3`, son espace d’observation et son espace d’action ;
- **PyTorch** permet de définir les réseaux Actor et Critic et de calculer automatiquement les gradients ;
- **NumPy** sert à manipuler les observations produites par l’environnement ;
- **Stable-Baselines3** est importé comme point de comparaison avec une implémentation industrielle de PPO, même si l’agent étudié ici est codé manuellement.

Cette séparation est importante pédagogiquement : nous utilisons une bibliothèque pour la simulation, mais nous construisons nous-mêmes les composants essentiels de PPO.

In [1]:
import random
from collections import deque

import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions.categorical as Categorical
import numpy as np

from stable_baselines3 import PPO, DQN, A2C, SAC


## 2. Découvrir l’environnement

Avant de choisir un algorithme, il faut identifier le contrat de l’environnement : quel état reçoit l’agent, quelles actions peut-il exécuter et comment l’environnement signale-t-il la fin d’un épisode ?

La cellule suivante crée une instance de `LunarLander-v3`. Le rendu visuel n’est pas activé pendant l’entraînement afin de conserver de bonnes performances.

In [2]:
env = gym.make("LunarLander-v3")

In [3]:
print(f"Espace d'observation: {env.observation_space}")

print(f"Espace d'action: {env.action_space}")

Espace d'observation: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Espace d'action: Discrete(4)


### 2.1. Actions, observations et récompenses

#### Espace d’action

`LunarLander-v3` possède un espace d’action discret de taille 4 :

| Action | Commande |
| ---: | --- |
| `0` | ne rien faire |
| `1` | allumer le moteur latéral gauche |
| `2` | allumer le moteur principal |
| `3` | allumer le moteur latéral droit |

Le choix d’un espace discret est déterminant : une distribution catégorielle suffit pour représenter la politique. L’Actor produira donc quatre logits, un par action, puis `Categorical` transformera ces scores en probabilités.

#### Espace d’observation

L’état est un vecteur de 8 valeurs :

1. position horizontale ;
2. position verticale ;
3. vitesse horizontale ;
4. vitesse verticale ;
5. angle de l’atterrisseur ;
6. vitesse angulaire ;
7. contact de la jambe gauche avec le sol ;
8. contact de la jambe droite avec le sol.

Le réseau reçoit donc un vecteur de dimension 8. Les deux dernières composantes sont des indicateurs de contact, tandis que les six premières décrivent la position et la dynamique du véhicule.

#### Récompense et fin d’épisode

La récompense totale d’un épisode est la somme des récompenses obtenues à chaque étape. Elle dépend notamment de la proximité de la plateforme, de la vitesse, de l’inclinaison, du contact des jambes et du carburant consommé.

Un atterrissage réussi apporte une récompense finale positive, tandis qu’un crash apporte une forte pénalité. Dans l’environnement standard, un score d’au moins 200 est généralement considéré comme une solution.

L’API récente de Gymnasium distingue `terminated` et `truncated` :

- `terminated` signifie que l’épisode s’est terminé naturellement, par réussite ou échec ;
- `truncated` signifie qu’une limite externe, comme la durée maximale, a interrompu l’épisode.

Dans le code, `done = terminated or truncated` permet de gérer les deux cas.

#### Transition physique

Sous le capot, Lunar Lander utilise Box2D pour simuler la physique. L’agent n’a pas accès aux équations internes : il observe seulement $s_t$, choisit $a_t$, puis reçoit $(r_{t+1}, s_{t+1})$. Cette interaction définit la transition :

$$s_t \xrightarrow{a_t} (r_{t+1}, s_{t+1})$$

## 3. Pourquoi choisir PPO ?

Lunar Lander possède des états continus, mais un nombre fini d’actions. Un algorithme de policy gradient est bien adapté : au lieu d’apprendre uniquement la valeur des actions, il apprend directement une politique probabiliste.

Nous choisissons **PPO** pour trois raisons :

- il réutilise plusieurs fois une même collecte de trajectoires, ce qui améliore l’efficacité des données ;
- il limite l’amplitude des mises à jour et réduit le risque de dégrader brutalement la politique ;
- il fonctionne bien avec un Actor-Critic et des espaces d’action discrets.

PPO s’appuie sur l’idée d’Actor-Critic :

- l’**Actor** choisit les actions ;
- le **Critic** estime la valeur des états et fournit un signal d’apprentissage.

PPO ne correspond pas simplement à A2C avec un nom différent : il reprend une logique Actor-Critic, mais ajoute un objectif probabiliste avec ratio et clipping pour contrôler la mise à jour.

In [4]:
class Actor(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(8, 64),
            nn.Tanh(),
            nn.Linear(64, 4)
        )

    def forward(self, state):
        logits = self.net(state)
        return Categorical.Categorical(logits=logits)



class Critic(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(8, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        return self.net(state)

## 4. Construire l’Actor et le Critic

Les deux réseaux utilisent une couche cachée de 64 neurones et une activation `Tanh`. Ce choix est courant pour les problèmes de contrôle : la fonction est bornée et l’approximation reste régulière lorsque les observations changent progressivement.

### Actor

L’Actor reçoit un état de dimension 8 et renvoie quatre logits. Ces logits ne sont pas encore des probabilités ; `Categorical(logits=logits)` construit une distribution catégorielle en appliquant implicitement une normalisation de type softmax.

Pendant la collecte, `dist.sample()` introduit de l’exploration. Pendant l’évaluation, `dist.probs.argmax()` choisit au contraire l’action la plus probable de manière déterministe.

### Critic

Le Critic reçoit le même état, mais renvoie une seule valeur scalaire :

$$V(s_t) = \mathbb{E}\left[\sum_{k=0}^{\infty} \gamma^k r_{t+k+1} \mid s_t\right]$$

Il ne choisit aucune action. Il sert de référence pour déterminer si une action a été meilleure ou moins bonne que prévu.

In [5]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []

    def add(self, state, action, log_prob, reward, done):
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.dones.append(done)

    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.log_probs.clear()
        self.rewards.clear()
        self.dones.clear()
        
    def get_tensors(self, device="cpu"):
 
        states = torch.tensor(self.states, dtype=torch.float32, device=device)
        actions = torch.tensor(self.actions, dtype=torch.long, device=device)
        log_probs = torch.tensor(self.log_probs, dtype=torch.float32, device=device)
        rewards = torch.tensor(self.rewards, dtype=torch.float32, device=device)
        dones = torch.tensor(self.dones, dtype=torch.float32, device=device)
  

        return states, actions, log_probs, rewards, dones

## 5. Collecter une expérience avec le Rollout Buffer

PPO est un algorithme **on-policy** : les données utilisées pour mettre à jour l’Actor doivent provenir de la politique actuelle. Le buffer ne conserve donc pas les expériences indéfiniment comme un replay buffer de DQN.

Pour chaque étape, nous mémorisons :

- l’état `state` ;
- l’action choisie ;
- la log-probabilité de cette action selon l’ancienne politique ;
- la récompense ;
- le marqueur de fin d’épisode.

La log-probabilité est conservée car PPO comparera ensuite l’ancienne politique à la nouvelle. Après la mise à jour, les transitions sont supprimées avec `buffer.clear()` et une nouvelle collecte est réalisée avec la politique actualisée.

La méthode `get_tensors` convertit les listes Python en tenseurs PyTorch afin que les opérations sur tout le rollout soient vectorisées.

In [13]:
def compute_gae(
    states,
    rewards,
    dones,
    critic,
    next_state,
    gamma=0.99, #Importance du futur
    lam=0.95    #Importance des erreurs
):
    
    with torch.no_grad():
        values = critic(states).squeeze(-1)

        bosstrap_value = critic(next_state).squeeze(-1)

    advantages = []
    gae = 0.0

    for t in reversed(range(len(rewards))):

        if dones[t]:
            next_value = torch.tensor(0.0)
        else:
            if t == len(rewards)-1:
                next_value = bosstrap_value
            else:
                next_value = values[t + 1]

        # δt​=rt​+γV(st+1​)−V(st​)
        delta = (
            rewards[t]
            + gamma * next_value
            - values[t]
        )
        
        gae = delta + gamma * lam * (1 - dones[t]) * gae

        advantages.append(gae)
    
    advantages.reverse()

    return (torch.stack(advantages) - torch.stack(advantages).mean() ) / ( torch.stack(advantages).std() + 1e-8)

## 6. Estimer l’avantage avec GAE

L’avantage mesure si une action a produit un résultat meilleur ou pire que la valeur attendue par le Critic :

$$A_t = Q(s_t,a_t) - V(s_t)$$

Dans la pratique, nous ne connaissons pas directement $Q(s_t,a_t)$. La méthode **Generalized Advantage Estimation (GAE)** construit une approximation à partir des erreurs temporelles :

$$\delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)$$

Puis elle accumule ces erreurs vers le passé :

$$\hat{A}_t = \delta_t + \gamma \lambda (1-d_t)\hat{A}_{t+1}$$

- $\gamma$ contrôle l’importance des récompenses futures ;
- $\lambda$ contrôle le compromis entre biais et variance ;
- $d_t$ vaut 1 lorsque l’épisode est terminé.

Le parcours en sens inverse est nécessaire, car l’avantage à l’instant $t$ dépend de l’estimation calculée à l’instant suivant. La normalisation finale des avantages facilite l’optimisation en gardant des valeurs centrées et d’échelle comparable.

Le dernier état du rollout est utilisé pour **bootstrapper** la valeur si la trajectoire s’arrête uniquement parce que le rollout de 2048 étapes est terminé.

In [14]:
def ppo_update(
    actor:Actor,
    critic:Critic,
    optimizer_actor,
    optimizer_critic,
    states,
    actions,
    old_log_probs,
    advantages,
    returns,
    clip_epsilon=0.2
):

    # 1. Nouvelle politique

    dist = actor(states)
    new_log_probs = dist.log_prob(actions)

   
    # 2. Ratio
    # e(log(old) - log(new)) = old/new

    ratio = torch.exp(
        new_log_probs - old_log_probs
    )


    # 3. PPO CLIP


    unclipped = ratio * advantages

    clipped = torch.clamp(
        ratio,
        1 - clip_epsilon,
        1 + clip_epsilon
    ) * advantages

    actor_loss = -torch.min(
        unclipped,
        clipped
    ).mean()

    
    # 4. Critic

    values = critic(states).squeeze(-1)

    critic_loss = F.mse_loss(
        values,
        returns
    )

    # 5. Update Actor


    optimizer_actor.zero_grad()

    actor_loss.backward()

    optimizer_actor.step()


    # 6. Update Critic

    optimizer_critic.zero_grad()

    critic_loss.backward()

    optimizer_critic.step()

    return actor_loss.item(), critic_loss.item()

## 7. Mettre à jour PPO

La mise à jour PPO compare la nouvelle politique à celle qui a généré les données. Pour une action observée, le ratio est :

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{old}}(a_t \mid s_t)} = \exp\left(\log \pi_\theta - \log \pi_{\theta_{old}}\right)$$

Sans contrainte, l’optimiseur pourrait modifier trop fortement la politique en une seule mise à jour. PPO borne donc le ratio :

$$\operatorname{clip}(r_t, 1-\epsilon, 1+\epsilon)$$

L’objectif de l’Actor utilise le minimum entre la version non bornée et la version bornée :

$$L^{CLIP} = \mathbb{E}_t\left[\min\left(r_t A_t, \operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t\right)\right]$$

Dans le code, la perte est négative car PyTorch minimise une fonction alors que l’objectif PPO doit être maximisé.

Le Critic est entraîné séparément avec une erreur quadratique entre sa valeur prédite et le retour cible :

$$L_V = \left(V(s_t) - R_t\right)^2$$

Les deux optimiseurs sont distincts afin de mettre à jour séparément la politique et l’estimateur de valeur.

In [15]:
def collect_rollout(
    env:gym.Env,
    actor:Actor,
    buffer:RolloutBuffer,
    rollout_steps=2048
):

    state, _ = env.reset()

    for _ in range(rollout_steps):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        # Actor
        dist = actor(state_tensor)

        action = dist.sample()

        log_prob = dist.log_prob(action)

        # Environment
        next_state, reward, terminated, truncated, _ = env.step(
            action.item()
        )

        done = terminated or truncated

        # Buffer
        buffer.add(
            state,
            action.item(),
            log_prob.item(),
            reward,
            done
        )

        state = next_state

        if done:
            state, _ = env.reset()

    return torch.Tensor(state)

## 8. Configurer et entraîner l’agent

Nous créons un environnement, un Actor, un Critic, un buffer et deux optimiseurs Adam. L’Actor et le Critic ont des paramètres différents, donc chacun possède son propre optimiseur.

Le choix `batch_size = 64` signifie qu’un rollout de 2048 transitions est découpé en 32 mini-lots. Chaque mini-lot est parcouru une fois dans cette implémentation avant d’être supprimé.

La boucle d’entraînement suit ce cycle :

1. collecter 2048 transitions avec la politique courante ;
2. calculer les avantages avec GAE ;
3. construire les retours cibles du Critic ;
4. mélanger les indices pour éviter un apprentissage dans l’ordre temporel ;
5. mettre à jour l’Actor et le Critic sur les mini-lots ;
6. vider le buffer et recommencer.

Le message affiché à chaque update contient `Actor Loss` et `Critic Loss`. Ces pertes servent à suivre l’optimisation, mais elles ne mesurent pas directement la qualité des atterrissages : la récompense moyenne et le taux de réussite restent les indicateurs principaux.

In [16]:
# Hyperparametres
batch_size = 64

# Environnement
env = gym.make("LunarLander-v3")

# Actor et critique

actor = Actor()
critic = Critic()

# Buffer
buffer = RolloutBuffer()

# Les optimiseurs
optimizer_actor = torch.optim.Adam(actor.parameters(),lr=1e-3)
optimizer_critic = torch.optim.Adam(critic.parameters(),lr=1e-3)

In [17]:
for update in range(1000):
    # Collecte

    next_state = collect_rollout(
        env,
        actor,
        buffer,
        rollout_steps=2048
    )

    # Récupération des tenseurs
    
    states , actions , old_log_probs , rewards , dones = buffer.get_tensors()


    # GAE


    advantages = compute_gae(
        states,
        rewards,
        dones,
        critic,
        next_state
    )

    # Returns

    with torch.no_grad():

        values = critic(states).squeeze(-1)

        returns = advantages + values


    # PPO UPDATE
    # Au lieu d'utiliser nos 2048 collecte pour modfier et ensuite jeter on passe dessus 4 fois avant de jeter (2048 / 64 = 32 minis batchs)
    
    indices = torch.randperm(len(states))
    
    for start in range(0, len(states), batch_size):

        batch_indices = indices[start:start + batch_size]

        batch_states = states[batch_indices]
        batch_actions = actions[batch_indices]
        batch_old_log_probs = old_log_probs[batch_indices]
        batch_advantages = advantages[batch_indices]
        batch_returns = returns[batch_indices]
        
        actor_loss, critic_loss = ppo_update(
            actor,
            critic,
            optimizer_actor,
            optimizer_critic,
            batch_states,
            batch_actions,
            batch_old_log_probs,
            batch_advantages,
            batch_returns
        )

    # Reset


    buffer.clear()

    print(
        f"Update {update} | "
        f"Actor Loss: {actor_loss:.3f} | "
        f"Critic Loss: {critic_loss:.3f}"
    )

Update 0 | Actor Loss: -0.003 | Critic Loss: 0.906
Update 1 | Actor Loss: 0.144 | Critic Loss: 0.822
Update 2 | Actor Loss: 0.017 | Critic Loss: 0.929
Update 3 | Actor Loss: -0.344 | Critic Loss: 0.658
Update 4 | Actor Loss: -0.105 | Critic Loss: 0.461
Update 5 | Actor Loss: -0.127 | Critic Loss: 0.388
Update 6 | Actor Loss: 0.069 | Critic Loss: 0.627
Update 7 | Actor Loss: -0.190 | Critic Loss: 1.102
Update 8 | Actor Loss: -0.079 | Critic Loss: 0.577
Update 9 | Actor Loss: 0.108 | Critic Loss: 0.602
Update 10 | Actor Loss: 0.090 | Critic Loss: 0.765
Update 11 | Actor Loss: -0.123 | Critic Loss: 0.634
Update 12 | Actor Loss: 0.100 | Critic Loss: 0.691
Update 13 | Actor Loss: -0.022 | Critic Loss: 0.644
Update 14 | Actor Loss: -0.140 | Critic Loss: 1.017
Update 15 | Actor Loss: -0.008 | Critic Loss: 0.623
Update 16 | Actor Loss: 0.103 | Critic Loss: 1.018
Update 17 | Actor Loss: -0.034 | Critic Loss: 0.601
Update 18 | Actor Loss: -0.160 | Critic Loss: 0.750
Update 19 | Actor Loss: -0.11

## 9. Évaluer les performances

L’entraînement et l’évaluation doivent être séparés. Pendant la collecte, l’Actor échantillonne ses actions pour explorer. Pour évaluer la politique, nous choisissons systématiquement l’action la plus probable.

Une seule partie ne permet pas de conclure : le point de départ et la dynamique peuvent varier. Nous allons donc mesurer la politique sur plusieurs épisodes indépendants et calculer :

- la récompense moyenne et médiane ;
- l’écart-type, qui mesure la variabilité ;
- la meilleure et la pire récompense ;
- le nombre moyen d’étapes ;
- le taux d’atterrissage réussi.

Dans `LunarLander-v3`, une réussite naturelle est signalée par `terminated` et une interruption par `truncated`. Il faut conserver cette distinction pour ne pas confondre une réussite avec une fin due à la limite de temps.

In [34]:
n_eval_episodes = 100
eval_env = gym.make("LunarLander-v3")
eval_rewards = []
eval_steps = []
successes = 0

actor.eval()
try:
    for episode in range(n_eval_episodes):
        state, _ = eval_env.reset()
        total_reward = 0.0
        steps = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            state_tensor = torch.tensor(
                state,
                dtype=torch.float32
            ).unsqueeze(0)

            with torch.no_grad():
                distribution = actor(state_tensor)
                action = distribution.probs.argmax(dim=1).item()

            state, reward, terminated, truncated, _ = eval_env.step(action)
            total_reward += reward
            steps += 1

        eval_rewards.append(total_reward)
        eval_steps.append(steps)
        successes += int(total_reward >= 200)
finally:
    eval_env.close()

rewards_array = np.array(eval_rewards, dtype=np.float32)
steps_array = np.array(eval_steps, dtype=np.int32)

print(f"Épisodes évalués       : {n_eval_episodes}")
print(f"Récompense moyenne     : {rewards_array.mean():.2f}")
print(f"Récompense médiane     : {np.median(rewards_array):.2f}")
print(f"Écart-type             : {rewards_array.std():.2f}")
print(f"Meilleure récompense   : {rewards_array.max():.2f}")
print(f"Pire récompense        : {rewards_array.min():.2f}")
print(f"Étapes moyennes        : {steps_array.mean():.2f}")
print(f"Atterrissages réussis  : {successes}/{n_eval_episodes}")
print(f"Taux de réussite       : {successes / n_eval_episodes:.1%}")

Épisodes évalués       : 100
Récompense moyenne     : 57.12
Récompense médiane     : 67.52
Écart-type             : 97.97
Meilleure récompense   : 213.95
Pire récompense        : -136.48
Étapes moyennes        : 731.33
Atterrissages réussis  : 5/100
Taux de réussite       : 5.0%


### Résultats de cette session

L’évaluation déterministe porte sur 100 épisodes. Les actions sont choisies avec `argmax`, mais les épisodes peuvent encore différer car l’environnement initialise les états de manière aléatoire.

| Indicateur | Résultat |
| --- | ---: |
| Récompense moyenne | `57,12` |
| Récompense médiane | `67,52` |
| Écart-type | `97,97` |
| Meilleure récompense | `213,95` |
| Pire récompense | `-136,48` |
| Nombre moyen d’étapes | `731,33` |
| Atterrissages réussis | `5/100` |
| Taux de réussite | `5 %` |

Le meilleur score dépasse le seuil de 200, ce qui montre que l’agent est capable de réussir au moins certains épisodes. Cependant, le taux de réussite de `5 %` est faible et l’écart-type de `97,97` révèle une forte variabilité. La médiane de `67,52` indique que le comportement typique est meilleur que l’échec complet, mais encore loin d’une politique fiable.

**Conclusion de la session :** l’agent PPO apprend des comportements partiellement utiles, mais il ne résout pas encore Lunar Lander de manière robuste. Il faut suivre l’évolution de ces indicateurs au fil des updates et répéter l’évaluation avec plusieurs graines avant de conclure à une amélioration stable.

Une bonne politique devrait augmenter progressivement la récompense moyenne et le taux de réussite tout en réduisant la dispersion des scores. Les pertes `Actor Loss` et `Critic Loss` sont utiles pour diagnostiquer l’optimisation, mais elles ne suffisent pas à juger la qualité des atterrissages.

## 10. Sauvegarder les réseaux

Nous sauvegardons séparément l’Actor et le Critic dans le dossier `models/`.

- l’Actor contient la politique utilisée pour choisir les actions ;
- le Critic contient l’estimateur de valeur utilisé pendant l’apprentissage.

Pour un projet durable, sauvegarder les `state_dict()` ainsi que les hyperparamètres est généralement plus robuste que sauvegarder directement l’objet Python complet. Ici, la sauvegarde de l’objet rend toutefois la démonstration simple à recharger dans le même environnement.

In [33]:
torch.save(actor, "models/lunarlanderdiscret-actor.pt")
torch.save(critic, "models/lunarlanderdiscret-critic.pt")


## 11. Observer une partie en direct

Le mode `human` ouvre un rendu visuel de l’environnement. L’agent est évalué sans échantillonnage : `argmax` sélectionne l’action ayant la probabilité la plus élevée.

Cette cellule sert de contrôle qualitatif. Elle permet de voir si l’atterrisseur corrige sa position, réduit sa vitesse et utilise les moteurs de manière cohérente. Le score imprimé est utile, mais l’image permet aussi de repérer un comportement instable ou une oscillation excessive.

In [31]:
env = gym.make(
    "LunarLander-v3",
    render_mode="human"
)

try:
    state, _ = env.reset()

    done = False
    total_reward = 0

    while not done:

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        with torch.no_grad():
            dist = actor(state_tensor)
            action = dist.probs.argmax().item()

        next_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        state = next_state
        total_reward += reward

        print("Reward :", total_reward)
finally:
    env.close()

Reward : 3.6100205244750727
Reward : 1.703434954528615
Reward : 5.257336413968426
Reward : 3.3640285471022597
Reward : 6.537617837710804
Reward : 4.774276265582535
Reward : 8.113675779165408
Reward : 6.422368120541638
Reward : 4.741089919669168
Reward : 8.445287503166526
Reward : 6.692593703456822
Reward : 9.55057826588573
Reward : 8.40283141104315
Reward : 9.45770995487012
Reward : 9.884588066146398
Reward : 13.126228174076846
Reward : 11.5695524787664
Reward : 13.810294254064019
Reward : 12.1793365010368
Reward : 14.338652207155587
Reward : 12.758108679179895
Reward : 15.287441369324704
Reward : 15.515231406259094
Reward : 15.654173389689404
Reward : 17.099285637730244
Reward : 15.600098493146216
Reward : 16.40953485941143
Reward : 16.735341631209156
Reward : 17.956367930623003
Reward : 21.77255725496455
Reward : 20.524532563083863
Reward : 19.24603201972739
Reward : 23.261815046111103
Reward : 21.851800665738033
Reward : 23.330817285115703
Reward : 25.66666920354676
Reward : 24.0323

## 12. Enregistrer une vidéo

`RecordVideo` utilise le mode `rgb_array` pour capturer les images produites par l’environnement et les encoder dans `./videos`.

Cette cellule reprend la politique gloutonne de l’évaluation, mais ajoute l’enregistrement. Le `try/finally` garantit que l’environnement est fermé même si une erreur survient pendant l’exécution.

In [32]:
from gymnasium.wrappers import RecordVideo

env = gym.make("LunarLander-v3", render_mode="rgb_array")

env = RecordVideo(
    env, 
    video_folder="./videos", 
    episode_trigger=lambda ep_id: True,
    name_prefix="lunarlander-discret"
)

state, _ = env.reset()
done = False

try:
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            dist = actor(state_tensor)
            action = dist.probs.argmax().item()

        next_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        state = next_state

finally:

    env.close()

print("Vidéo enregistrée dans le dossier ./videos")

e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Vidéo enregistrée dans le dossier ./videos


## 13. Bilan et pistes d’amélioration

Ce notebook a suivi toute la chaîne d’un agent PPO : perception de l’état, distribution d’actions, collecte on-policy, estimation GAE, clipping de la politique, optimisation séparée de l’Actor et du Critic, puis évaluation.

La conclusion doit être fondée sur les scores obtenus pendant l’évaluation multi-épisodes, et non uniquement sur la diminution des pertes. Une politique intéressante combine :

- une récompense moyenne élevée ;
- un taux d’atterrissage réussi élevé ;
- une médiane proche de la moyenne ;
- un écart-type limité.

Pour améliorer l’expérience, on peut ensuite :

- tracer la récompense moyenne par update ;
- utiliser plusieurs environnements en parallèle ;
- ajouter une entropie dans la perte de l’Actor pour maintenir l’exploration ;
- séparer explicitement les transitions tronquées des épisodes réellement terminés ;
- comparer les résultats avec `stable_baselines3.PPO` ;
- sauvegarder les paramètres, les hyperparamètres et les scores d’évaluation dans un fichier de suivi.